In [1]:
from dotenv import load_dotenv
import os
from accounts import Account
from agents.mcp import MCPServerStdio
from accounts_client import get_accounts_tools_openai, read_accounts_resource, list_accounts_tools
from agents import Agent, trace, Runner
from IPython.display import display, Markdown

In [2]:
load_dotenv(override=True)

True

In [3]:
account = Account.get("Prosen")

In [4]:
account

Account(name='prosen', balance=9741.484, strategy='', holdings={'AMZN': 6}, transactions=[3 shares of AMZN at 42.084 each., 3 shares of AMZN at 44.088 each.], portfolio_value_time_series=[('2026-04-08 19:06:07', 10149.748), ('2026-04-08 19:09:02', 9942.748), ('2026-04-10 16:38:40', 9969.484), ('2026-04-10 16:38:42', 9969.484)])

In [5]:
account.buy_shares(symbol="AMZN", quantity=3, rationale="Because the company is doing great as a cloud provider")

'Completed. Latest details:\n{"name": "prosen", "balance": 9567.136, "strategy": "", "holdings": {"AMZN": 9}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 42.084, "timestamp": "2026-04-08 19:06:07", "rationale": "Because the company is doing great as a cloud provider"}, {"symbol": "AMZN", "quantity": 3, "price": 44.088, "timestamp": "2026-04-10 16:38:40", "rationale": "Because the company is doing great as a cloud provider"}, {"symbol": "AMZN", "quantity": 3, "price": 58.116, "timestamp": "2026-04-10 17:18:38", "rationale": "Because the company is doing great as a cloud provider"}], "portfolio_value_time_series": [["2026-04-08 19:06:07", 10149.748], ["2026-04-08 19:09:02", 9942.748], ["2026-04-10 16:38:40", 9969.484], ["2026-04-10 16:38:42", 9969.484], ["2026-04-10 17:18:38", 10314.136]], "total_portfolio_value": 10314.136, "total_profit_loss": 314.1360000000004}'

In [6]:
account.report()

'{"name": "prosen", "balance": 9567.136, "strategy": "", "holdings": {"AMZN": 9}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 42.084, "timestamp": "2026-04-08 19:06:07", "rationale": "Because the company is doing great as a cloud provider"}, {"symbol": "AMZN", "quantity": 3, "price": 44.088, "timestamp": "2026-04-10 16:38:40", "rationale": "Because the company is doing great as a cloud provider"}, {"symbol": "AMZN", "quantity": 3, "price": 58.116, "timestamp": "2026-04-10 17:18:38", "rationale": "Because the company is doing great as a cloud provider"}], "portfolio_value_time_series": [["2026-04-08 19:06:07", 10149.748], ["2026-04-08 19:09:02", 9942.748], ["2026-04-10 16:38:40", 9969.484], ["2026-04-10 16:38:42", 9969.484], ["2026-04-10 17:18:38", 10314.136], ["2026-04-10 17:18:39", 9846.136]], "total_portfolio_value": 9846.136, "total_profit_loss": -153.86399999999958}'

In [7]:
account.list_transactions()

[{'symbol': 'AMZN',
  'quantity': 3,
  'price': 42.084,
  'timestamp': '2026-04-08 19:06:07',
  'rationale': 'Because the company is doing great as a cloud provider'},
 {'symbol': 'AMZN',
  'quantity': 3,
  'price': 44.088,
  'timestamp': '2026-04-10 16:38:40',
  'rationale': 'Because the company is doing great as a cloud provider'},
 {'symbol': 'AMZN',
  'quantity': 3,
  'price': 58.116,
  'timestamp': '2026-04-10 17:18:38',
  'rationale': 'Because the company is doing great as a cloud provider'}]

In [8]:
params = {"command": "/home/prosen/.local/share/pipx/venvs/uv/bin/uv", "args": ["run", "accounts_server.py"]}

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()

In [9]:
mcp_tools

[Tool(name='get_balance', title=None, description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'get_balanceOutput', 'type': 'object'}, icons=None, annotations=None, meta=None),
 Tool(name='get_holdings', title=None, description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, outputSchema={'additionalProperties': {'type': 'integer'}, 'title': 'get_holdingsDictOutput', 'type': 'object'}, icons=None, annotations=None, meta=None),
 Tool(name='buy_shares', 

In [10]:
instructions = "You are able to manage an account for a client, and answer questions about the account."
request = "My name is Prosen and my account is under the name Prosen. What's my balance and my holdings?"
model = "gpt-4.1-mini"

In [11]:
params = {"command": "/home/prosen/.local/share/pipx/venvs/uv/bin/uv", "args": ["run", "accounts_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(name="account_manager", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("account_manager"):
        result = await Runner.run(starting_agent=agent, input=request)
    display(Markdown(result.final_output))

Prosen, your account balance is $9,567.14. Your current holdings include 9 shares of Amazon (AMZN). Is there anything specific you would like to do with your account?

In [12]:
mcp_tools = await list_accounts_tools()
print(mcp_tools)
openai_tools = await get_accounts_tools_openai()
print(openai_tools)

[Tool(name='get_balance', title=None, description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'get_balanceOutput', 'type': 'object'}, icons=None, annotations=None, meta=None), Tool(name='get_holdings', title=None, description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, outputSchema={'additionalProperties': {'type': 'integer'}, 'title': 'get_holdingsDictOutput', 'type': 'object'}, icons=None, annotations=None, meta=None), Tool(name='buy_shares', ti

In [14]:
request = "My name is Prosen and my account is under the name Prosen. What's my balance?"

with trace("account_mcp_client"):
    agent = Agent(name="account_manager", instructions=instructions, model=model, tools=openai_tools)
    result = await Runner.run(starting_agent=agent, input=request)
    display(Markdown(result.final_output))

Your current account balance is $9,567.14. Is there anything specific you would like to do with your account?

In [15]:
context = await read_accounts_resource("Prosen")
print(context)

{"name": "prosen", "balance": 9567.136, "strategy": "", "holdings": {"AMZN": 9}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 42.084, "timestamp": "2026-04-08 19:06:07", "rationale": "Because the company is doing great as a cloud provider"}, {"symbol": "AMZN", "quantity": 3, "price": 44.088, "timestamp": "2026-04-10 16:38:40", "rationale": "Because the company is doing great as a cloud provider"}, {"symbol": "AMZN", "quantity": 3, "price": 58.116, "timestamp": "2026-04-10 17:18:38", "rationale": "Because the company is doing great as a cloud provider"}], "portfolio_value_time_series": [["2026-04-08 19:06:07", 10149.748], ["2026-04-08 19:09:02", 9942.748], ["2026-04-10 16:38:40", 9969.484], ["2026-04-10 16:38:42", 9969.484], ["2026-04-10 17:18:38", 10314.136], ["2026-04-10 17:18:39", 9846.136], ["2026-04-10 17:23:43", 10026.136]], "total_portfolio_value": 10026.136, "total_profit_loss": 26.136000000000422}


In [16]:
Account.get("Prosen").report()

'{"name": "prosen", "balance": 9567.136, "strategy": "", "holdings": {"AMZN": 9}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 42.084, "timestamp": "2026-04-08 19:06:07", "rationale": "Because the company is doing great as a cloud provider"}, {"symbol": "AMZN", "quantity": 3, "price": 44.088, "timestamp": "2026-04-10 16:38:40", "rationale": "Because the company is doing great as a cloud provider"}, {"symbol": "AMZN", "quantity": 3, "price": 58.116, "timestamp": "2026-04-10 17:18:38", "rationale": "Because the company is doing great as a cloud provider"}], "portfolio_value_time_series": [["2026-04-08 19:06:07", 10149.748], ["2026-04-08 19:09:02", 9942.748], ["2026-04-10 16:38:40", 9969.484], ["2026-04-10 16:38:42", 9969.484], ["2026-04-10 17:18:38", 10314.136], ["2026-04-10 17:18:39", 9846.136], ["2026-04-10 17:23:43", 10026.136], ["2026-04-10 17:24:31", 9972.136]], "total_portfolio_value": 9972.136, "total_profit_loss": -27.863999999999578}'